# Laboratorium 5 (4 pkt)

Celem czwartego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmów głębokiego uczenia aktywnego. Zaimplementowane algorytmy będą testowane z wykorzystaniem środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [ ]:
from collections import deque
import gym
import numpy as np
import random
from copy import deepcopy

Dołączenie bibliotek do obsługi sieci neuronowych

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(342)

class DQN(nn.Module):
    def __init__(self, state_size, action_size, hidden_neurons, learning_rate):
        super(DQN, self).__init__()

        self.fc1 = nn.Linear(state_size, hidden_neurons)
        self.fc2 = nn.Linear(hidden_neurons, hidden_neurons)
        self.out = nn.Linear(hidden_neurons, action_size)

        self.learning_rate = learning_rate
        self.optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        return self.out(x)
    
    def predict(self, state):
        state = torch.FloatTensor(state)
        with torch.no_grad():
            q_values = self.forward(state)
        
        return q_values.numpy()
    
    def fit(self, states, targets):
        if len(states) == 0:
            return
        
        states = np.array(states)
        targets = np.array(targets)
        
        if states.ndim == 1:
            states = states.reshape(1, -1)
        if targets.ndim == 1:
            targets = targets.reshape(1, -1)

        states = torch.FloatTensor(states)
        targets = torch.FloatTensor(targets)

        self.optimizer.zero_grad()
        outputs = self.forward(states)
        loss = F.smooth_l1_loss(outputs, targets)
        loss.backward()
        self.optimizer.step()

## Zadanie 1 - Double Deep Q-Network

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu Double Deep Q-Network. Wartoscią oczekiwaną sieci jest:
\begin{equation}
       Q^*(s, a) \approx r + \gamma argmax_{a'}Q_\theta'(s', a') 
\end{equation}
a wagi pomiędzy sieciami wymieniane są co dziesięć aktualizacji wag sieci sterującej poczynaniami agenta ($Q$).
</p>

In [ ]:
class DDQNAgent:
    def __init__(self, state_size, action_size, model_init):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95    # discount rate
        self.epsilon = 0.5  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.95
        self.learning_rate = 0.001
        self.model = self._build_model(model_init)
        self.target_model = self._build_model(model_init)
        self.update_weights()
        self.replay_counter = 1

    def _build_model(self, model_init):
        return deepcopy(model_init)
        
    def remember(self, state, action, reward, next_state, done):
        #Function adds information to the memory about last action and its results
        self.memory.append((state, action, reward, next_state, done)) 

    def get_action(self, state):
        """
        Compute the action to take in the current state, including exploration.
        With probability self.epsilon, we should take a random action.
            otherwise - the best policy action (self.get_best_action).

        Note: To pick randomly from a list, use random.choice(list).
              To pick True or False with a given probablity, generate uniform number in [0, 1]
              and compare it with your probability
        """

        if random.uniform(0, 1) < self.epsilon:
            chosen_action = random.choice(range(self.action_size))
        else:
            chosen_action = self.get_best_action(state)
        
        return chosen_action

  
    def get_best_action(self, state):
        """
        Compute the best action to take in a state.
        """
        q_values = self.model.predict(state) 
        
        best_value = np.max(q_values)
        best_actions = np.where(q_values == best_value)[0]
        best_action = random.choice(best_actions)
        
        return best_action

    def replay(self, batch_size):
        """
        Function learn network using randomly selected actions from the memory. 
        First calculates Q value for the next state and choose action with the biggest value.
        Target value is calculated according to:
                Q(s,a) := (r + gamma * max_a(Q(s', a)))
        except the situation when the next action is the last action, in such case Q(s, a) := r.
        In order to change only those weights responsible for chosing given action, the rest values should be those
        returned by the network for state state.
        The network should be trained on batch_size samples.
        After each 10 Q Network trainings parameters should be copied to the target Q Network
        """
        if len(self.memory) == 0:
            return
        
        batch = random.sample(self.memory, min(batch_size, len(self.memory)))
        states, targets = [], []

        for state, action, reward, next_state, done in batch:
            target = self.model.predict(state)
            if done:
                target[action] = reward
            else:
                next_q_values = self.target_model.predict(next_state)
                best_next_action = np.argmax(self.model.predict(next_state))
                target[action] = reward + self.gamma * next_q_values[best_next_action]
            
            states.append(state)
            targets.append(target)

        self.model.fit(states, targets)


    def update_epsilon_value(self):
        #Every each epoch epsilon value should be updated according to equation: 
        #self.epsilon *= self.epsilon_decay, but the updated value shouldn't be lower then epsilon_min value
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def update_weights(self):
        """copy trained Q Network params to target Q Network"""
        
        self.target_model = deepcopy(self.model)
        


Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [ ]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
learning_rate = 0.001

model = DQN(state_size, action_size, hidden_neurons=64, learning_rate=learning_rate)

Czas nauczyć agenta gry w środowisku *CartPool*:

In [ ]:
agent = DDQNAgent(action_size, learning_rate, model)

agent.epsilon = 0.75

done = False
batch_size = 64
EPISODES = 1000
counter = 0
for e in range(EPISODES):
    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()
    
        state = env_state
        
        for time in range(500):
            action = agent.get_action(state)
            next_state_env, reward, done, _ = env.step(action)
            total_reward += reward

            next_state = next_state_env

            #add to experience memory
            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break

            agent.replay(batch_size)
        
        summary.append(total_reward)

    agent.update_epsilon_value()
        
    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))    
    
    if np.mean(summary) > 195:
        print ("You Win!")
        break
